# Fine-Tuning FLAN-T5-small for Information-Preserving Extreme Text Summarization

**Google Colab step-by-step notebook**

This notebook fine-tunes `google/flan-t5-small` on your custom dataset to summarize paragraphs **within a requested maximum length** of 10%, 20%, or 30% of the source.

## Experiment design

- **Baseline:** original `google/flan-t5-small` with no additional fine-tuning.
- **Fine-tuned model:** the same checkpoint further fine-tuned on your custom examples.
- **Input format:** `summarize to at most 10%: [source paragraph]`
- **Target:** human-written summary that is **at most** the requested length. The percentage is a **ceiling**, not an exact target: a summary may be shorter than its cap, but never longer. (An exact ratio is not achievable for every paragraph, because some paragraphs cannot be summarized well at an exact length.)
- **Dataset plan:** 500 training source paragraphs, 50 validation paragraphs, and 100 test paragraphs. Each source has three target summaries.

This produces 1,500 training records, 150 validation records, and 300 test records if each source has all three target lengths.

> Important: Split by **original source paragraph** before expanding to three target-length rows. Never let different summaries of the same source appear in different splits.

## Notebook flow

| Steps | What happens |
|---|---|
| 0-1 | Runtime, configuration, and data upload |
| **2-5** | **Dataset, tokenizer, and truncation checks, ending in a pre-training gate that blocks training if any blocking check fails** |
| 6-9 | Tokenize, configure, train, and save the model |
| 10-13 | Generate predictions, compression ratio, ROUGE-L, and paired bootstrap comparison |
| 14-18 | Blind human annotation, final metrics with confidence intervals, plot, and reporting notes |

All settings you might want to change are in one **configuration cell** (Step 0).

## Step 0 — Set up the Colab runtime and configuration

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU). A GPU is required in practice; CPU training will be extremely slow. The pre-training gate (Step 5) blocks training on CPU unless you set `REQUIRE_GPU = False`.

This notebook expects three CSV files:
- `train.csv`
- `validation.csv`
- `test.csv`

Each CSV must contain these columns:

| source | target_length | target_summary |
|---|---:|---|
| Original paragraph | 10 | Human-written summary of at most 10% of the source words |
| Original paragraph | 20 | Human-written summary of at most 20% of the source words |
| Original paragraph | 30 | Human-written summary of at most 30% of the source words |

`target_length` is the **maximum** allowed percentage. Reference summaries do not need to hit it exactly, but they must not exceed it (the dataset check enforces this).

The `source` text must be **identical** across the three rows of the same paragraph. A `source_id` is generated automatically.

The `test.csv` file must also contain the human-written reference summaries for all three target lengths.

**Precision note:** T5-family models can produce NaN losses under fp16. This notebook therefore trains in full fp32 on a T4 and switches to bf16 only on GPUs that support it (A100, L4, and newer). flan-t5-small is small enough that fp32 is fine.

In [ ]:
# Install the latest available package versions.
%pip install -q -U transformers datasets accelerate sentencepiece rouge-score pandas scikit-learn matplotlib

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import set_seed

from google.colab import files

# ======================= Configuration (edit here) =======================
SEED = 42
MODEL_NAME = "google/flan-t5-small"

MAX_INPUT_LENGTH = 512     # tokens (prompt + source)
MAX_TARGET_LENGTH = 128    # tokens; also used as max_new_tokens at generation time

LEARNING_RATE = 3e-4       # common starting point for T5-small; try 5e-5 as a comparison run
NUM_EPOCHS = 5
BATCH_SIZE = 8

# ---- Dataset-check settings ----
EXPECTED_SOURCES = {"train": 500, "validation": 50, "test": 100}
CEILING_SLACK_PP = 1.0        # percentage points a summary may exceed its cap (absorbs word-count rounding)
NEAR_DUP_THRESHOLD = 0.90     # TF-IDF cosine similarity that flags near-duplicate sources across splits
TRUNCATION_TOLERANCE = 0.01   # max fraction of rows allowed to be truncated by the token limits
ALLOW_TRUNCATION = False      # True turns truncation failures into warnings
STRICT_UNKNOWN_CHARS = True   # True: characters the tokenizer cannot represent block training
REQUIRE_GPU = True            # True: a missing GPU blocks training
# =========================================================================

set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training is blocked at Step 5 unless REQUIRE_GPU = False.")

## Step 1 — Upload the dataset CSV files

Upload `train.csv`, `validation.csv`, and `test.csv` when prompted. If your files are already in Google Drive, you can replace this upload cell with Drive paths.

In [ ]:
uploaded = files.upload()

required_files = ["train.csv", "validation.csv", "test.csv"]
missing = [name for name in required_files if name not in uploaded]
if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Upload all three CSV files."
    )

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
display(train_df.head())

## Step 2 — Dataset checks (run before any training)

Steps 2-4 build a list of check results. Each check is **✅ pass**, **⚠️ warn** (review, but training is allowed), or **❌ fail** (blocks training at the Step 5 gate).

**2a — Structure** (stops immediately if broken): required columns, missing values, valid `target_length` labels, empty text. Also assigns a `source_id` to every paragraph.

**2b — Content checks:**
- expected number of sources per split (500 / 50 / 100) and 3 rows per source
- duplicate `(source, target_length)` rows
- every source has all of 10, 20, and 30
- **exact source overlap** between splits (leakage)
- **near-duplicate sources** across splits (TF-IDF similarity) — inspect any hits manually
- reference summaries shorter than the source
- **every reference summary is within its cap** (at most 10%, 20%, or 30% of the source words, plus `CEILING_SLACK_PP`); the table also shows how much of the allowed budget the references use
- summary lengths do not shrink as the cap grows (10% cap <= 20% cap <= 30% cap) for each source
- very short sources

In [ ]:
# ---------- Helpers (used throughout the notebook) ----------
CHECK_RESULTS = []

def record(name, status, message=""):
    assert status in {"pass", "warn", "fail"}
    CHECK_RESULTS.append((name, status, message))
    icon = {"pass": "✅", "warn": "⚠️", "fail": "❌"}[status]
    print(f"{icon} {name}" + (f" — {message}" if message else ""))

def word_count(text):
    # Numbers like 5,000 or 3.5 and words like well-known / don't count as one word.
    # "PHP 5,000" also counts as one word, so rewriting "₱5,000" as "PHP 5,000" in the
    # normalization step does not change any compression ratio.
    return len(re.findall(r"(?:PHP\s(?=\d))?\w+(?:[-'.,]\w+)*", str(text)))

def compression_ratio(source, summary):
    source_words = word_count(source)
    if source_words == 0:
        return 0.0
    return 100 * word_count(summary) / source_words

def build_prompt(source, target_length):
    return f"summarize to at most {int(target_length)}%: {source}"

SPLITS = {"train": train_df, "validation": val_df, "test": test_df}
ID_PREFIX = {"train": "tr", "validation": "va", "test": "te"}
REQUIRED_COLUMNS = ["source", "target_length", "target_summary"]

# ---------- 2a. Structure (raises immediately: later checks depend on it) ----------
for split_name, df in SPLITS.items():
    missing_cols = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing_cols:
        raise ValueError(f"{split_name}.csv is missing columns: {missing_cols}")

    na_counts = df[REQUIRED_COLUMNS].isna().sum()
    if na_counts.any():
        raise ValueError(f"{split_name}.csv has missing values:\n{na_counts[na_counts > 0]}")

    for col in ["source", "target_summary"]:
        df[col] = (
            df[col].astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

    lengths = pd.to_numeric(df["target_length"], errors="raise")
    if not (lengths % 1 == 0).all():
        raise ValueError(f"{split_name}: target_length must be whole numbers.")
    df["target_length"] = lengths.astype(int)

    if not set(df["target_length"].unique()).issubset({10, 20, 30}):
        raise ValueError(
            f"{split_name}: target_length must be 10, 20, or 30. "
            f"Found {sorted(int(v) for v in df['target_length'].unique())}."
        )
    if (df["source"] == "").any() or (df["target_summary"] == "").any():
        raise ValueError(f"{split_name}.csv contains empty text.")

    # One id per unique source paragraph (e.g. tr_0001); used to pair results later.
    df["source_id"] = (
        ID_PREFIX[split_name] + "_"
        + (df.groupby("source", sort=False).ngroup() + 1).astype(str).str.zfill(4)
    )

CHECK_RESULTS.clear()
record("Required columns, no missing values, valid labels, no empty text", "pass")

# ---------- 2b. Sizes, duplicates, completeness ----------
for split_name, df in SPLITS.items():
    n_sources = df["source"].nunique()
    expected = EXPECTED_SOURCES[split_name]
    if n_sources == expected and len(df) == 3 * expected:
        record(f"[{split_name}] size matches plan", "pass", f"{n_sources} sources, {len(df)} rows")
    else:
        record(f"[{split_name}] size matches plan", "warn",
               f"{n_sources} sources / {len(df)} rows (plan: {expected} / {3 * expected}). "
               "Fine if intentional; update EXPECTED_SOURCES to silence this.")

    dup_rows = df[df.duplicated(["source", "target_length"], keep=False)]
    if len(dup_rows):
        record(f"[{split_name}] no duplicate (source, target_length) rows", "fail",
               f"{len(dup_rows)} rows involved, e.g. source_ids {dup_rows['source_id'].unique()[:5].tolist()}")
    else:
        record(f"[{split_name}] no duplicate (source, target_length) rows", "pass")

    lengths_per_source = df.groupby("source")["target_length"].apply(lambda x: tuple(sorted(set(x))))
    incomplete = lengths_per_source[lengths_per_source.apply(lambda t: t != (10, 20, 30))]
    if len(incomplete):
        record(f"[{split_name}] every source has 10%, 20%, and 30% targets", "fail",
               f"{len(incomplete)} sources incomplete")
    else:
        record(f"[{split_name}] every source has 10%, 20%, and 30% targets", "pass")

# ---------- 2c. Exact source leakage between splits ----------
src = {name: set(df["source"]) for name, df in SPLITS.items()}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = src[a] & src[b]
    if overlap:
        record(f"No exact source overlap: {a} vs {b}", "fail",
               f"{len(overlap)} shared paragraphs. Split by source first, then expand to 10/20/30 rows.")
    else:
        record(f"No exact source overlap: {a} vs {b}", "pass")

# ---------- 2d. Near-duplicate sources between splits ----------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

uniq = {name: df.drop_duplicates("source")[["source_id", "source"]].reset_index(drop=True)
        for name, df in SPLITS.items()}
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(
    pd.concat([u["source"] for u in uniq.values()])
)

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    sim = cosine_similarity(vectorizer.transform(uniq[a]["source"]), vectorizer.transform(uniq[b]["source"]))
    ii, jj = np.where(sim >= NEAR_DUP_THRESHOLD)
    if len(ii):
        examples = [(uniq[a]["source_id"][i], uniq[b]["source_id"][j], round(float(sim[i, j]), 2))
                    for i, j in list(zip(ii, jj))[:5]]
        record(f"No near-duplicate sources: {a} vs {b}", "warn",
               f"{len(ii)} pairs with similarity >= {NEAR_DUP_THRESHOLD}; inspect manually, e.g. {examples}")
    else:
        record(f"No near-duplicate sources: {a} vs {b}", "pass")

In [ ]:
# ---------- 2e. Reference summaries versus their 10/20/30 caps ----------
# The label is a MAXIMUM: a reference may be shorter than its cap, but not longer.

def check_reference_caps(split_name, df):
    df["source_words"] = df["source"].map(word_count)
    df["summary_words"] = df["target_summary"].map(word_count)
    df["ref_ratio"] = [compression_ratio(s, t) for s, t in zip(df["source"], df["target_summary"])]

    too_long = df[df["ref_ratio"] >= 100]
    if len(too_long):
        record(f"[{split_name}] reference summaries shorter than source", "fail",
               f"{len(too_long)} rows where the summary is as long as the source")
    else:
        record(f"[{split_name}] reference summaries shorter than source", "pass")

    within_cap = df["ref_ratio"] <= df["target_length"] + CEILING_SLACK_PP
    over_cap = df[~within_cap]
    check_name = f"[{split_name}] reference summaries within their cap (+{CEILING_SLACK_PP:g} pp slack)"
    if len(over_cap):
        worst = over_cap.assign(over=over_cap["ref_ratio"] - over_cap["target_length"]).nlargest(3, "over")
        examples = [f"{r.source_id} ({int(r.target_length)}% cap -> {r.ref_ratio:.1f}%)" for r in worst.itertuples()]
        record(check_name, "fail",
               f"{len(over_cap)} of {len(df)} rows exceed their cap, e.g. {examples}. "
               "Shorten them, or raise CEILING_SLACK_PP if this is only word-count rounding.")
    else:
        record(check_name, "pass")

    stats = df.groupby("target_length")["ref_ratio"].agg(["count", "mean", "std", "min", "max"])
    stats["mean_cap_use_%"] = 100 * stats["mean"] / stats.index.to_numpy()
    stats["within_cap_%"] = 100 * within_cap.groupby(df["target_length"]).mean()
    return stats.round(1)

ratio_tables = []

for split_name, df in SPLITS.items():
    stats = check_reference_caps(split_name, df)
    stats.insert(0, "split", split_name)
    ratio_tables.append(stats.reset_index())

    # A larger cap should not force a shorter summary: 10% cap <= 20% cap <= 30% cap (in words)
    words = df.pivot_table(index="source_id", columns="target_length", values="summary_words", aggfunc="first").dropna()
    bad_order = words[~((words[10] <= words[20]) & (words[20] <= words[30]))]
    if len(bad_order):
        record(f"[{split_name}] summary length does not shrink as the cap grows", "warn",
               f"{len(bad_order)} sources have a longer summary for a smaller cap, e.g. {bad_order.index[:5].tolist()}")
    else:
        record(f"[{split_name}] summary length does not shrink as the cap grows", "pass")

    # Very short sources make a 10% cap only a few words long
    short_sources = df.drop_duplicates("source_id").query("source_words < 30")
    if len(short_sources):
        record(f"[{split_name}] sources of at least 30 words", "warn",
               f"{len(short_sources)} sources are under 30 words, e.g. {short_sources['source_id'].head(5).tolist()}")
    else:
        record(f"[{split_name}] sources of at least 30 words", "pass")

print("\nReference compression ratio by split and cap (percent of source words).")
print("mean_cap_use_% = average ratio as a share of the cap; within_cap_% = share of references at or under the cap.")
display(pd.concat(ratio_tables, ignore_index=True))

## Step 3 — Load the pretrained FLAN-T5-small model

This is a pretrained, instruction-tuned model. Your experiment evaluates whether **additional fine-tuning on your custom dataset** improves its performance on your summarization task. It is not training a Transformer from scratch.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Loaded:", MODEL_NAME)
print("Device:", device)

## Step 4 — Tokenizer and truncation checks

**4a — Characters the tokenizer cannot represent.** T5's vocabulary has no entry for some characters (for example `{`, `}`, `<`, some curly quotes, and possibly `₱`). Such a character becomes `<unk>` and is silently dropped from the model's output, which can erase facts such as peso amounts for **both** models. This step finds every unrepresentable character, rewrites the ones listed in `NORMALIZE_MAP` (for example `₱5,000` becomes `PHP 5,000`) consistently across all three splits, and blocks training if any remain.

Edit `NORMALIZE_MAP` if your data has other problem characters.

**4b — Round trip.** Encodes and decodes every text and reports where the tokenizer changes the content.

**4c — Truncation.** Token limits cut text silently. Truncated *inputs* hide facts from the model, and compression ratio is still computed against the full source. Truncated *targets* teach the model to produce unfinished summaries. Generation is also capped by `MAX_TARGET_LENGTH`.

In [ ]:
# ---------- 4a. Characters the tokenizer cannot represent ----------
NORMALIZE_MAP = {
    "₱": "PHP ",
    "\u2018": "'", "\u2019": "'",      # curly single quotes
    "\u201c": '"', "\u201d": '"',      # curly double quotes
    "\u2013": "-", "\u2014": "-",      # en / em dash
    "\u2026": "...",                   # ellipsis
    "\u00a0": " ", "\u200b": "",       # non-breaking space, zero-width space
}

TEXT_COLUMNS = ["source", "target_summary"]

def all_texts():
    return [t for df in SPLITS.values() for c in TEXT_COLUMNS for t in df[c].tolist()]

def find_unknown_chars(texts):
    chars = sorted(set("".join(texts)) - {" "})
    return [ch for ch in chars
            if tokenizer.unk_token_id in tokenizer(ch, add_special_tokens=False)["input_ids"]]

def describe_chars(chars, texts):
    joined = " ".join(texts)
    for ch in chars:
        i = joined.index(ch)
        print(f"   {ch!r} (U+{ord(ch):04X}) x{joined.count(ch)}   e.g. ...{joined[max(0, i - 20):i + 20]}...")

unknown = find_unknown_chars(all_texts())
if unknown:
    print("Characters the tokenizer cannot represent:")
    describe_chars(unknown, all_texts())

fixable = {ch: NORMALIZE_MAP[ch] for ch in unknown if ch in NORMALIZE_MAP}
if fixable:
    for df in SPLITS.values():
        for col in TEXT_COLUMNS:
            for ch, replacement in fixable.items():
                df[col] = df[col].str.replace(ch, replacement, regex=False)
            df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()
    print("Applied the same normalization to train, validation, and test:", fixable)

    # Word counts changed (e.g. "₱5,000" -> "PHP 5,000"), so re-run the cap checks.
    # The gate uses the latest result for each check.
    print("Re-checking reference caps after normalization:")
    for split_name, df in SPLITS.items():
        check_reference_caps(split_name, df)

still_unknown = find_unknown_chars(all_texts())

if not unknown:
    record("All characters are representable by the tokenizer", "pass")
elif not still_unknown:
    record("All characters are representable by the tokenizer", "pass",
           f"normalized {fixable} (same rewrite applied to every split)")
else:
    print("Still unrepresentable after normalization:")
    describe_chars(still_unknown, all_texts())
    record("All characters are representable by the tokenizer",
           "fail" if STRICT_UNKNOWN_CHARS else "warn",
           f"{still_unknown} would become <unk>. Add them to NORMALIZE_MAP or remove them from the data.")

# ---------- 4b. Encode/decode round trip ----------
def first_diff(a, b, ctx=25):
    for i, (x, y) in enumerate(zip(a, b)):
        if x != y:
            return a[max(0, i - ctx):i + ctx], b[max(0, i - ctx):i + ctx]
    return a[-2 * ctx:], b[-2 * ctx:]

def strip_ws(t):
    return re.sub(r"\s+", "", t)

mismatches = []
for text in sorted(set(all_texts())):
    decoded = tokenizer.decode(tokenizer(text)["input_ids"], skip_special_tokens=True)
    if strip_ws(decoded) != strip_ws(text):
        mismatches.append((text, decoded))

if mismatches:
    for text, decoded in mismatches[:3]:
        before, after = first_diff(strip_ws(text), strip_ws(decoded))
        print(f"   original: ...{before}...\n   decoded:  ...{after}...")
    record("Tokenizer round trip preserves text", "warn",
           f"{len(mismatches)} of {len(set(all_texts()))} unique texts change after encode/decode "
           "(see examples above). Inspect for lost characters or digits.")
else:
    record("Tokenizer round trip preserves text", "pass")

In [ ]:
# ---------- 4c. Truncation by token limits ----------
trunc_rows = []
for split_name, df in SPLITS.items():
    prompts = [build_prompt(s, l) for s, l in zip(df["source"], df["target_length"])]
    input_tokens = np.array([len(ids) for ids in tokenizer(prompts)["input_ids"]])
    target_tokens = np.array([len(ids) for ids in tokenizer(text_target=df["target_summary"].tolist())["input_ids"]])
    trunc_rows.append({
        "split": split_name,
        "input_max": int(input_tokens.max()),
        "input_p95": int(np.percentile(input_tokens, 95)),
        "input_truncated_%": round(100 * float((input_tokens > MAX_INPUT_LENGTH).mean()), 2),
        "target_max": int(target_tokens.max()),
        "target_p95": int(np.percentile(target_tokens, 95)),
        "target_truncated_%": round(100 * float((target_tokens > MAX_TARGET_LENGTH).mean()), 2),
    })

trunc_df = pd.DataFrame(trunc_rows)
print(f"Token lengths versus limits (MAX_INPUT_LENGTH={MAX_INPUT_LENGTH}, MAX_TARGET_LENGTH={MAX_TARGET_LENGTH}):")
display(trunc_df)

problems = []
for r in trunc_rows:
    if r["input_truncated_%"] > 100 * TRUNCATION_TOLERANCE:
        suggested = int(np.ceil(r["input_max"] / 64) * 64)
        problems.append(f"{r['split']}: {r['input_truncated_%']}% of inputs exceed MAX_INPUT_LENGTH (try {suggested})")
    if r["target_truncated_%"] > 100 * TRUNCATION_TOLERANCE:
        suggested = int(np.ceil(r["target_max"] / 32) * 32)
        problems.append(f"{r['split']}: {r['target_truncated_%']}% of targets exceed MAX_TARGET_LENGTH (try {suggested})")

if problems:
    record("Token limits do not truncate the data",
           "warn" if ALLOW_TRUNCATION else "fail",
           "; ".join(problems) + ". Raise the limits in Step 0 (and re-run from Step 0), or set ALLOW_TRUNCATION = True.")
else:
    record("Token limits do not truncate the data", "pass")

# ---------- Look at one tokenized example ----------
example = train_df.sample(1, random_state=SEED).iloc[0]
example_prompt = build_prompt(example["source"], example["target_length"])
example_input = tokenizer(example_prompt, truncation=True, max_length=MAX_INPUT_LENGTH)["input_ids"]
example_label = tokenizer(text_target=example["target_summary"], truncation=True, max_length=MAX_TARGET_LENGTH)["input_ids"]
print("\nExample", example["source_id"])
print("PROMPT :", example_prompt[:300], "...")
print("INPUT  :", len(example_input), "tokens ->", tokenizer.decode(example_input, skip_special_tokens=True)[:300], "...")
print("LABEL  :", len(example_label), "tokens ->", tokenizer.decode(example_label, skip_special_tokens=True))
print("TARGET :", example["target_summary"])

## Step 5 — Pre-training gate

This cell adds the GPU check, prints every check result, saves the report as `dataset_checks_report.csv`, and **raises an error if any check failed**. Training cells refuse to run until this gate passes.

Warnings do not block training, but read them: they flag things worth mentioning in your methodology.

If a check fails, fix the CSVs (or the settings in Step 0), then re-run from Step 1 (data problems) or Step 0 (setting problems).

In [ ]:
if torch.cuda.is_available():
    record("GPU available", "pass", torch.cuda.get_device_name(0))
else:
    record("GPU available", "fail" if REQUIRE_GPU else "warn",
           "No GPU detected. Use Runtime > Change runtime type > T4 GPU, or set REQUIRE_GPU = False (very slow).")

report_df = (
    pd.DataFrame(CHECK_RESULTS, columns=["check", "status", "message"])
    .drop_duplicates("check", keep="last")
    .reset_index(drop=True)
)
report_df.to_csv("/content/dataset_checks_report.csv", index=False)

n_fail = int((report_df["status"] == "fail").sum())
n_warn = int((report_df["status"] == "warn").sum())
n_pass = int((report_df["status"] == "pass").sum())

print(f"\nPRE-TRAINING GATE: {n_pass} passed, {n_warn} warnings, {n_fail} failed")
if n_warn or n_fail:
    display(report_df[report_df["status"] != "pass"])

PRETRAIN_OK = n_fail == 0
files.download("/content/dataset_checks_report.csv")

if not PRETRAIN_OK:
    raise RuntimeError(
        f"{n_fail} blocking check(s) failed (see the table above). Fix them before training."
    )
print("All blocking checks passed. Review any warnings above, then continue to training.")

## Step 6 — Convert rows into model inputs

Each input includes the requested **maximum** length (for example `summarize to at most 20%:`). The corresponding human-written summary is the target output.

`MAX_INPUT_LENGTH` and `MAX_TARGET_LENGTH` are token limits set in Step 0 and checked for truncation in Step 4. The cell refuses to run if the pre-training gate has not passed.

In [ ]:
assert globals().get("PRETRAIN_OK", False), "Run the pre-training gate (Step 5) and fix any failed checks first."

def preprocess_batch(examples):
    prompts = [
        build_prompt(source, length)
        for source, length in zip(examples["source"], examples["target_length"])
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target_summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

tokenized_train = train_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=val_dataset.column_names
)

print("Tokenized training records:", len(tokenized_train))
print("Tokenized validation records:", len(tokenized_val))

## Step 7 — Configure fine-tuning

These are starting hyperparameters for a small experiment, not guaranteed optimal settings. `LEARNING_RATE`, `NUM_EPOCHS`, and `BATCH_SIZE` are set in Step 0.

- **Learning rate:** 3e-4 is a common starting point for T5-small. If validation loss rises after the first epoch or two, lower it (for example 1e-4 or 5e-5). If it is still falling at the last epoch, train longer or raise it.
- **Precision:** fp32 on a T4 (T5 can overflow in fp16); bf16 automatically on GPUs that support it.
- **Memory:** if Colab runs out of GPU memory, reduce `BATCH_SIZE` to 4 or 2.

The trainer keeps the checkpoint with the lowest validation loss.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100   # padded label positions are ignored by the loss
)

# bf16 needs compute capability >= 8 (A100, L4, ...). A T4 is 7.5, so it trains in fp32.
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/flan-t5-extreme-summary",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,

    weight_decay=0.01,
    num_train_epochs=NUM_EPOCHS,

    predict_with_generate=False,
    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=False,
    bf16=USE_BF16,
    report_to="none",
    seed=SEED
)

print("Learning rate:", LEARNING_RATE, "| epochs:", NUM_EPOCHS, "| batch size:", BATCH_SIZE)
print("Precision:", "bf16" if USE_BF16 else "fp32")

## Step 8 — Fine-tune the model

Run the first cell to begin training. The model compares its predicted tokens with the target summaries and updates its parameters to reduce the training loss.

The second cell plots training and validation loss and reports which epoch was selected. Validation loss is useful for checkpoint selection, but it does not replace the four final research metrics.

In [ ]:
assert globals().get("PRETRAIN_OK", False), "Run the pre-training gate (Step 5) and fix any failed checks first."

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator
)

train_result = trainer.train()
print(train_result)

In [ ]:
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
log_df.to_csv("/content/training_log.csv", index=False)

train_curve = log_df.dropna(subset=["loss"]) if "loss" in log_df else pd.DataFrame()
eval_curve = log_df.dropna(subset=["eval_loss"]) if "eval_loss" in log_df else pd.DataFrame()

plt.figure(figsize=(8, 4.5))
if len(train_curve):
    plt.plot(train_curve["epoch"], train_curve["loss"], label="training loss", alpha=0.8)
if len(eval_curve):
    plt.plot(eval_curve["epoch"], eval_curve["eval_loss"], marker="o", label="validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("Selected checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)
if len(eval_curve):
    best_epoch = float(eval_curve.loc[eval_curve["eval_loss"].idxmin(), "epoch"])
    print("Best epoch:", best_epoch, "of", NUM_EPOCHS)
    if best_epoch >= NUM_EPOCHS:
        print("Validation loss was still improving at the last epoch: consider more epochs or a higher learning rate.")
    elif best_epoch <= 1:
        print("Validation loss was lowest in the first epoch: the model may be overfitting; try a lower learning rate.")

files.download("/content/training_log.csv")

## Step 9 — Save and download the fine-tuned model

The model and tokenizer are saved together so they can be loaded later for inference. Because `load_best_model_at_end=True`, this saves the best checkpoint, not necessarily the last epoch.

In [ ]:
FINAL_MODEL_DIR = "/content/final-finetuned-model"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Saved model to:", FINAL_MODEL_DIR)

# Package the model folder as a ZIP for download.
!zip -qr /content/final-finetuned-model.zip /content/final-finetuned-model
files.download("/content/final-finetuned-model.zip")

## Step 10 — Load both models for comparison

The baseline must be the original checkpoint, without additional fine-tuning. The fine-tuned model is the best checkpoint selected during training.

In [ ]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
baseline_model.eval()

fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_MODEL_DIR).to(device)
fine_tuned_model.eval()

tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)

print("Baseline and fine-tuned models loaded.")

## Step 11 — Generate summaries

The requested maximum length is given in the prompt. The model is not guaranteed to respect it; whether each summary stays within its cap, and by how much it overshoots, is measured later.

For a fair comparison, both models use the same prompts and the same generation settings (`GEN_KWARGS`). Generation is batched and sorted by length, which is much faster than one example at a time. Outputs are returned in the original order.

In [ ]:
GEN_KWARGS = dict(num_beams=4, do_sample=False)

def generate_summaries(sources, target_lengths, generation_model, batch_size=16):
    prompts = [build_prompt(s, l) for s, l in zip(sources, target_lengths)]
    order = np.argsort([len(p) for p in prompts])[::-1]    # longest first
    outputs = [None] * len(prompts)

    generation_model.eval()
    for start in tqdm(range(0, len(order), batch_size), leave=False):
        idx = order[start:start + batch_size]
        batch = tokenizer(
            [prompts[i] for i in idx],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH
        ).to(device)

        with torch.no_grad():
            output_ids = generation_model.generate(
                **batch,
                max_new_tokens=MAX_TARGET_LENGTH,
                **GEN_KWARGS
            )

        for i, text in zip(idx, tokenizer.batch_decode(output_ids, skip_special_tokens=True)):
            outputs[i] = text.strip()

    return outputs

# Quick sanity check on one test paragraph.
sample_source = test_df.iloc[0]["source"]
sample_lengths = [10, 20, 30]
baseline_sample = generate_summaries([sample_source] * 3, sample_lengths, baseline_model)
finetuned_sample = generate_summaries([sample_source] * 3, sample_lengths, fine_tuned_model)

for pct, base_text, ft_text in zip(sample_lengths, baseline_sample, finetuned_sample):
    print(f"\n--- Requested length: {pct}% ---")
    print("BASELINE:")
    print(base_text)
    print("FINE-TUNED:")
    print(ft_text)

## Step 12 — Generate predictions for the complete test set

This produces six outputs per source paragraph: three length caps for the baseline and three for the fine-tuned model.

The generated results are saved to a CSV file (with `source_id`) for evaluation and manual annotation.

In [ ]:
test_sources = test_df["source"].tolist()
test_lengths = test_df["target_length"].astype(int).tolist()

baseline_predictions = generate_summaries(test_sources, test_lengths, baseline_model)
finetuned_predictions = generate_summaries(test_sources, test_lengths, fine_tuned_model)

predictions_df = pd.DataFrame({
    "source_id": test_df["source_id"].tolist(),
    "source": test_sources,
    "target_length": test_lengths,
    "reference_summary": test_df["target_summary"].tolist(),
    "baseline_summary": baseline_predictions,
    "finetuned_summary": finetuned_predictions
})

predictions_path = "/content/test_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

display(predictions_df.head(10))
files.download(predictions_path)

## Step 13 — Compression ratio, ROUGE-L, and paired bootstrap

Compression ratio is calculated from word counts. Because the requested length is a **maximum**, three measures describe length behavior:

- **Actual compression ratio:** generated words as a percentage of source words.
- **Within-target rate:** the share of summaries at or below their cap (allowing `CEILING_SLACK_PP`).
- **Mean overshoot:** how many percentage points a summary goes above its cap (0 when within it).

A model that ignores the cap can score well on fact coverage simply by writing longer summaries; these measures expose that. ROUGE-L F1 compares each generated summary with the corresponding human-written reference.

ROUGE-L measures textual overlap, not factual correctness. Fact coverage and factual consistency are annotated separately in the next steps.

The tables report the **mean and standard deviation** for each model and length. The paired bootstrap then compares the two models on the *same* source paragraphs: it resamples source paragraphs 10,000 times and reports the 95% confidence interval for the difference (fine-tuned minus baseline). If the interval includes 0, the data do not show a reliable difference. With about 100 test sources, a single run can otherwise mistake noise for improvement.

In [ ]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

metric_rows = []
for row in predictions_df.itertuples(index=False):
    for model_name, generated in [
        ("baseline", row.baseline_summary),
        ("fine_tuned", row.finetuned_summary)
    ]:
        ratio = compression_ratio(row.source, generated)
        metric_rows.append({
            "source_id": row.source_id,
            "model": model_name,
            "target_length": row.target_length,
            "source_words": word_count(row.source),
            "summary_words": word_count(generated),
            "compression_ratio_percent": ratio,
            "within_target_percent": 100.0 * (ratio <= row.target_length + CEILING_SLACK_PP),
            "overshoot_pp": max(0.0, ratio - row.target_length),
            "rougeL_f1": rouge.score(row.reference_summary, generated)["rougeL"].fmeasure
        })

metrics_df = pd.DataFrame(metric_rows)

summary_metrics = metrics_df.groupby(["model", "target_length"]).agg(
    actual_compression_percent=("compression_ratio_percent", "mean"),
    actual_compression_std=("compression_ratio_percent", "std"),
    within_target_percent=("within_target_percent", "mean"),
    mean_overshoot_pp=("overshoot_pp", "mean"),
    rougeL_f1=("rougeL_f1", "mean"),
    rougeL_f1_std=("rougeL_f1", "std")
).reset_index()

print("Model results (mean and std over test rows):")
display(summary_metrics.round(3))

print("\nFor reference, compression ratio of the human reference summaries in the test set (they only need to be at or below the cap):")
display(test_df.groupby("target_length")["ref_ratio"].agg(["mean", "std"]).round(2))


# ---------- Paired bootstrap helpers (reused in Step 16) ----------
def paired_bootstrap_ci(diffs, n_boot=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    diffs = np.asarray(diffs, dtype=float)
    idx = rng.integers(0, len(diffs), size=(n_boot, len(diffs)))
    boot_means = diffs[idx].mean(axis=1)
    low, high = np.percentile(boot_means, [2.5, 97.5])
    return diffs.mean(), low, high

def paired_comparison(df, value_col, label):
    rows = []
    for length, group in df.groupby("target_length"):
        wide = group.pivot(index="source_id", columns="model", values=value_col).dropna()
        diffs = (wide["fine_tuned"] - wide["baseline"]).to_numpy()
        mean_diff, low, high = paired_bootstrap_ci(diffs)
        rows.append({
            "metric": label,
            "target_length": length,
            "n_sources": len(diffs),
            "mean_diff_ft_minus_base": mean_diff,
            "ci95_low": low,
            "ci95_high": high,
            "ci_excludes_0": bool(low > 0 or high < 0)
        })
    return pd.DataFrame(rows)

rouge_paired = paired_comparison(metrics_df, "rougeL_f1", "ROUGE-L F1")
print("\nPaired bootstrap, ROUGE-L F1 (fine-tuned minus baseline):")
display(rouge_paired.round(4))

## Step 14 — Create the blind annotation sheet

Fact coverage and factual consistency require human review. To avoid bias toward whichever model you expect to win, the annotation sheet is **blinded**:

- The sheet **does not say which model wrote each summary**. For each source and length, the two outputs appear next to each other in random order.
- A separate **key file** (`annotation_key.csv`) maps each `annotation_id` back to the model. **Do not open the key until annotation is finished.**

Fill in these columns in `annotation_sheet_blind.csv`:

- `total_important_facts`: number of predefined important facts for that **source**. Define the fact list for each source **before reviewing any model output**, and enter the **same number on every row of that source** (the notebook checks this).
- `facts_preserved`: number of those facts accurately preserved in the generated summary.
- `no_factual_errors`: `1` if the summary contains no factual errors; otherwise `0`.

In [ ]:
long_rows = []
for row in predictions_df.itertuples(index=False):
    for model_name, generated in [
        ("baseline", row.baseline_summary),
        ("fine_tuned", row.finetuned_summary)
    ]:
        long_rows.append({
            "source_id": row.source_id,
            "model": model_name,
            "target_length": row.target_length,
            "source": row.source,
            "generated_summary": generated
        })

long_df = pd.DataFrame(long_rows)

# Group by source and length; randomize which model appears first within each pair.
rng = np.random.default_rng(SEED)
long_df["_order"] = rng.random(len(long_df))
long_df = (
    long_df.sort_values(["source_id", "target_length", "_order"])
    .drop(columns="_order")
    .reset_index(drop=True)
)
long_df["annotation_id"] = [f"A{i + 1:04d}" for i in range(len(long_df))]

key_df = long_df[["annotation_id", "model", "source_id", "target_length"]]
blind_df = long_df[["annotation_id", "source_id", "target_length", "source", "generated_summary"]].copy()
blind_df["total_important_facts"] = ""
blind_df["facts_preserved"] = ""
blind_df["no_factual_errors"] = ""

blind_df.to_csv("/content/annotation_sheet_blind.csv", index=False)
key_df.to_csv("/content/annotation_key.csv", index=False)

display(blind_df.head())
files.download("/content/annotation_sheet_blind.csv")
files.download("/content/annotation_key.csv")

## Step 15 — Calculate fact coverage and consistency rate

After completing `annotation_sheet_blind.csv`, upload it below. If this Colab session is still running, the key file is read from `/content/annotation_key.csv`; if the runtime was restarted, upload `annotation_key.csv` together with the completed sheet (any file with "key" in its name is treated as the key).

The cell checks that every annotation row is complete, values are in range, and the fact count is **identical for every row of the same source**.

In [ ]:
annotation_upload = files.upload()

csv_names = [name for name in annotation_upload if name.lower().endswith(".csv")]
key_names = [name for name in csv_names if "key" in name.lower()]
sheet_names = [name for name in csv_names if name not in key_names]

if len(sheet_names) != 1:
    raise FileNotFoundError("Upload exactly one completed annotation sheet CSV (plus the key CSV if the runtime restarted).")

key_path = key_names[0] if key_names else "/content/annotation_key.csv"
if not os.path.exists(key_path):
    raise FileNotFoundError("Key file not found. Upload annotation_key.csv together with the completed sheet.")

sheet_df = pd.read_csv(sheet_names[0])
key_df = pd.read_csv(key_path)

human_cols = ["total_important_facts", "facts_preserved", "no_factual_errors"]
missing_cols = {"annotation_id", *human_cols} - set(sheet_df.columns)
if missing_cols:
    raise ValueError(f"Annotation sheet is missing columns: {missing_cols}")

if sheet_df["annotation_id"].duplicated().any():
    raise ValueError("Duplicate annotation_id values in the annotation sheet.")
if set(sheet_df["annotation_id"]) != set(key_df["annotation_id"]):
    raise ValueError(
        "annotation_id values in the sheet do not match the key file "
        f"({len(sheet_df)} sheet rows vs {len(key_df)} key rows)."
    )

for col in human_cols:
    sheet_df[col] = pd.to_numeric(sheet_df[col], errors="coerce")

incomplete = sheet_df[sheet_df[human_cols].isna().any(axis=1)]
if len(incomplete):
    raise ValueError(
        f"{len(incomplete)} rows have blank or non-numeric annotation fields, "
        f"e.g. {incomplete['annotation_id'].head(5).tolist()}."
    )

if (sheet_df["total_important_facts"] <= 0).any():
    raise ValueError("total_important_facts must be greater than zero.")

if ((sheet_df["facts_preserved"] < 0) |
        (sheet_df["facts_preserved"] > sheet_df["total_important_facts"])).any():
    raise ValueError("facts_preserved must be between zero and total_important_facts.")

if not sheet_df["no_factual_errors"].isin([0, 1]).all():
    raise ValueError("no_factual_errors must contain only 0 or 1.")

# Re-attach the model labels and check the fact list is the same for every row of a source.
annotations_df = sheet_df[["annotation_id", *human_cols]].merge(
    key_df, on="annotation_id", how="inner", validate="one_to_one"
)

facts_per_source = annotations_df.groupby("source_id")["total_important_facts"].nunique()
inconsistent = facts_per_source[facts_per_source > 1]
if len(inconsistent):
    raise ValueError(
        f"total_important_facts differs between rows of the same source for {len(inconsistent)} sources, "
        f"e.g. {inconsistent.index[:5].tolist()}. Use one fact list per source for both models."
    )

annotations_df["fact_coverage_percent"] = (
    100 * annotations_df["facts_preserved"] / annotations_df["total_important_facts"]
)
annotations_df["consistency_rate_percent"] = 100 * annotations_df["no_factual_errors"]

human_metrics = annotations_df.groupby(["model", "target_length"]).agg(
    fact_coverage_percent=("fact_coverage_percent", "mean"),
    consistency_rate_percent=("consistency_rate_percent", "mean")
).reset_index()

display(human_metrics.round(3))

## Step 16 — Combine the four evaluation metrics

The final table reports the four metrics for each model and requested maximum length, with standard deviations for the two automatic metrics, plus the within-target rate and mean overshoot.

- Compression ratio: actual average percentage of source words.
- Within-target rate: percentage of summaries at or below their cap; mean overshoot: average percentage points above the cap.
- ROUGE-L F1: average overlap with the human reference.
- Fact coverage: average percentage of predefined facts preserved.
- Consistency rate: percentage of summaries with no factual errors.

The second table gives the **paired bootstrap comparison** (fine-tuned minus baseline) for all four metrics. `ci_excludes_0 = True` means the 95% interval does not include zero, so the difference is unlikely to be sampling noise. Compression ratio, within-target rate, and overshoot are included because a model that "improves" fact coverage only by writing longer summaries, or by exceeding the cap, has not really improved.

In [ ]:
final_results = summary_metrics.merge(
    human_metrics,
    on=["model", "target_length"],
    how="left"
)

final_results = final_results[
    [
        "model",
        "target_length",
        "actual_compression_percent",
        "actual_compression_std",
        "within_target_percent",
        "mean_overshoot_pp",
        "rougeL_f1",
        "rougeL_f1_std",
        "fact_coverage_percent",
        "consistency_rate_percent"
    ]
]

print("Final results by model and requested maximum length:")
display(final_results.round(3))
final_results.to_csv("/content/final_evaluation_results.csv", index=False)

paired_df = pd.concat([
    paired_comparison(metrics_df, "rougeL_f1", "ROUGE-L F1"),
    paired_comparison(metrics_df, "compression_ratio_percent", "Actual compression (%)"),
    paired_comparison(metrics_df, "within_target_percent", "Within-target rate (%)"),
    paired_comparison(metrics_df, "overshoot_pp", "Overshoot above cap (pp)"),
    paired_comparison(annotations_df, "fact_coverage_percent", "Fact coverage (%)"),
    paired_comparison(annotations_df, "consistency_rate_percent", "Consistency rate (%)"),
], ignore_index=True)

print("\nPaired bootstrap comparison, fine-tuned minus baseline (95% CI):")
display(paired_df.round(3))
paired_df.to_csv("/content/paired_comparison.csv", index=False)

files.download("/content/final_evaluation_results.csv")
files.download("/content/paired_comparison.csv")

## Step 17 — Plot fact coverage against actual compression ratio

This is the primary analysis proposed for the research: compare information preservation at different actual compression levels for the baseline and fine-tuned models. Each point is labeled with its requested maximum length. Because summaries only need to stay at or below the cap, actual compression can differ between models; compare fact coverage at similar actual compression, not just at the same cap.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for model_name, group in final_results.groupby("model"):
    group = group.sort_values("actual_compression_percent")
    plt.plot(
        group["actual_compression_percent"],
        group["fact_coverage_percent"],
        marker="o",
        label=model_name
    )
    for _, r in group.iterrows():
        plt.annotate(
            f"{int(r['target_length'])}%",
            (r["actual_compression_percent"], r["fact_coverage_percent"]),
            textcoords="offset points", xytext=(5, 5), fontsize=8
        )

plt.xlabel("Actual compression ratio (%)")
plt.ylabel("Fact coverage (%)")
plt.title("Fact Coverage vs. Actual Compression Ratio")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 18 — Save your experiment outputs

Download and retain:
- `dataset_checks_report.csv` (pre-training check results)
- `final-finetuned-model.zip`
- `training_log.csv`
- `test_predictions.csv`
- `annotation_sheet_blind.csv` (completed) and `annotation_key.csv`
- `final_evaluation_results.csv`
- `paired_comparison.csv`

## Interpretation and reporting

Use the results to answer:

**Does fine-tuning improve a pretrained Transformer's ability to preserve important facts when summarizing text to only 10%, 20%, or 30% of its original length?**

Compare the baseline and fine-tuned model at each length cap. Do not claim improvement based on ROUGE-L alone. Examine fact coverage and factual consistency alongside actual compression ratio, and use the paired bootstrap intervals: only describe a difference as an improvement when its interval excludes zero.

## Methodology details to report

- Dataset checks performed before training (leakage, duplicates, reference-ratio check) and any warnings that remained.
- Any text normalization applied to make the data representable by the tokenizer (for example `₱` rewritten as `PHP`), applied identically to all splits.
- Token limits used, and the fraction of data they truncated.
- Learning rate, epochs, batch size, precision (fp32 or bf16), and the epoch selected by validation loss.
- Annotation was blinded to model identity, with one fact list per source shared by both models.

## Experimental limitations to report

1. FLAN-T5-small is already instruction-tuned; the experiment measures the effect of additional task-specific fine-tuning.
2. The requested length is a maximum, not an exact target. A model may exceed the cap or use only a small part of it; the within-target rate, overshoot, and actual compression ratio are reported for this reason. Fact coverage naturally rises with summary length, so differences should be read together with actual compression.
3. ROUGE-L measures lexical sequence overlap and does not establish factual correctness.
4. Fact coverage and factual consistency depend on the quality and consistency of human annotations. A single annotator gives no measure of agreement; a second annotator on a subset would let you report it.
5. The dataset is relatively small. Results are from a single training run with one seed; the bootstrap captures test-sample variation but not training-run variation.
6. The bootstrap treats test source paragraphs as independent samples.